# Colab: MobileNetV4ConvSmall FSD50K Pretraining

This notebook pretrains a MobileNetV4 Conv Small audio classifier on FSD50K precomputed mel features (`npy_fsd50k.zip`) stored on Google Drive.

**Setup Instructions:**
1. Upload `npy_fsd50k.zip` to Google Drive
2. Keep `FSD50K.ground_truth` on Drive (for labels/splits)
3. Update the Drive path variables below if your layout is different
4. Ensure GPU is enabled: Runtime -> Change runtime type -> GPU
5. Run all cells in order

## Step 1: Mount Google Drive and Configure Paths

In [1]:
from pathlib import Path
import sys

In [2]:
PROJECT_ROOT = Path.cwd().resolve().parents[1]
ML_ROOT = PROJECT_ROOT / 'ml'

# Ground-truth CSV + vocabulary location (labels/splits)
FSD50K_GROUND_TRUTH_DIR = ML_ROOT / 'data'/ 'raw' / 'FSDK50_dataset' / 'FSD50K.ground_truth'

# Dev Patches path
FSD50K_FEATURE_DIR = ML_ROOT /'data' / 'npy_fsd50k_dev_patches'

# Eval Path
EVAL_NPY_FSD50K_DIR = ML_ROOT /'data' / 'npy_fsd50k_eval_patches'

# Output directories
MODEL_OUTPUT_DIR = ML_ROOT / 'export' / 'artifacts' / 'generated' / 'mobilenetv4_convsmall'
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert FSD50K_GROUND_TRUTH_DIR.exists(), f'FSD50K.ground_truth not found at {FSD50K_GROUND_TRUTH_DIR}'

npy_files = list(FSD50K_FEATURE_DIR.glob('*.npy'))
print(f'✓ Found {len(npy_files)} .npy feature files')



npy_files = list(EVAL_NPY_FSD50K_DIR.glob('*.npy'))
print(f'✓ Found {len(npy_files)} .npy eval feature files')

✓ Found 527797 .npy feature files
✓ Found 186456 .npy eval feature files


## Step 2: Install Dependencies

In [ ]:
%pip install -q tf-models-official tf-keras scikit-learn pandas numpy
print('Dependencies installed!')

In [ ]:
# Check GPU availability
import tensorflow as tf

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {tf.config.list_physical_devices("GPU")}')
if not tf.config.list_physical_devices('GPU'):
    print('⚠️ WARNING: No GPU detected. Training will be slow. Enable GPU in Runtime → Change runtime type')

## Step 3: Setup and Configuration

In [3]:
from __future__ import annotations

import json
import numpy as np
import pandas as pd
import tensorflow as tf
import tf_keras as keras
from official.vision.modeling.backbones import mobilenet
from sklearn.metrics import average_precision_score, f1_score, hamming_loss, label_ranking_average_precision_score, multilabel_confusion_matrix

# Runtime acceleration toggles
USE_MIXED_PRECISION = True
ENABLE_XLA = True

if ENABLE_XLA:
    tf.config.optimizer.set_jit(True)
if USE_MIXED_PRECISION:
    keras.mixed_precision.set_global_policy('mixed_float16')

# Training configuration
MODEL_ID = 'MobileNetV4ConvSmall'
SAMPLE_RATE = 16000
WINDOW_DURATION_MS = 960
WINDOW_SAMPLES = int(SAMPLE_RATE * WINDOW_DURATION_MS / 1000)

# Only the mel feature contract matters here; the notebook does not re-run preprocessing.
PREPROCESSING_CONFIG = {
    'stft': {'n_fft': 1024, 'hop_length': 512, 'win_length': 1024},
    'mel': {'n_mels': 96, 'power': 2.0},
}
EXPECTED_MEL_BINS = PREPROCESSING_CONFIG['mel']['n_mels']
TIME_FRAMES = 32
IMAGE_SIZE = (EXPECTED_MEL_BINS, TIME_FRAMES)
EMBEDDING_DIM = 512
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-5
PREDICTION_THRESHOLD = 0.5
AUTOTUNE = tf.data.AUTOTUNE

# Preprocessing summary for quick sanity checks in the notebook.
print('Configuration loaded')
print({
    'model_id': MODEL_ID,
    'sample_rate': SAMPLE_RATE,
    'preprocessing_config': PREPROCESSING_CONFIG,
    'image_size': IMAGE_SIZE,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'mixed_precision_policy': keras.mixed_precision.global_policy().name,
    'xla_enabled': ENABLE_XLA,
    'model_output_dir': str(MODEL_OUTPUT_DIR),
})


The dtype policy mixed_float16 may run slowly because this machine does not have a GPU. Only Nvidia GPUs with compute capability of at least 7.0 run quickly with mixed_float16.
If you will use compatible GPU(s) not attached to this host, e.g. by running a multi-worker model, you can ignore this warning. This message will only be logged once
Configuration loaded
{'model_id': 'MobileNetV4ConvSmall', 'sample_rate': 16000, 'preprocessing_config': {'stft': {'n_fft': 1024, 'hop_length': 512, 'win_length': 1024}, 'mel': {'n_mels': 96, 'power': 2.0}}, 'image_size': (96, 32), 'batch_size': 32, 'epochs': 20, 'mixed_precision_policy': 'mixed_float16', 'xla_enabled': True, 'model_output_dir': '/Users/oselumese/Documents/GitHub/SoundPulse/ml/export/artifacts/generated/mobilenetv4_convsmall'}


In [4]:
import numpy as np
from pathlib import Path

sample_feature_dir = FSD50K_FEATURE_DIR
expected_mels = PREPROCESSING_CONFIG['mel']['n_mels']

for feature_file in list(sample_feature_dir.glob('*.npy'))[:5]:
    feature = np.load(feature_file)
    print(feature_file.name, feature.shape, feature.dtype, feature.min(), feature.max(), np.mean(feature))
    if feature.ndim == 2 and expected_mels not in feature.shape:
        raise ValueError(
            f'{feature_file.name} does not match the expected mel-bin contract of {expected_mels} bins.'
        )


255902_032000.npy (96, 32) float16 -80.0 0.0 -57.12
145899_000000.npy (96, 32) float16 -80.0 0.0 -64.7
326392_088000.npy (96, 32) float16 -43.28 0.0 -23.78
95959_160000.npy (96, 32) float16 -80.0 0.0 -45.03
75435_024000.npy (96, 32) float16 -64.4 0.0 -28.58


## Step 4: Load FSD50K Dataset

In [5]:
def load_fsd50k_vocabulary(vocabulary_path: Path) -> pd.DataFrame:
    vocabulary = pd.read_csv(vocabulary_path, header=None, names=['index', 'label', 'mid'])
    vocabulary = vocabulary.sort_values('index').reset_index(drop=True)
    assert vocabulary['index'].tolist() == list(range(len(vocabulary))), 'FSD50K vocabulary indices must be contiguous.'
    return vocabulary


def labels_to_multihot(labels: str) -> np.ndarray:
    target = np.zeros(len(LABEL_ORDER), dtype=np.float32)
    for label in labels.split(','):
        index = LABEL_TO_INDEX.get(label)
        if index is not None:
            target[index] = 1.0
    return target


def load_fsd50k_split_patches(csv_path: Path, data_root: Path, split_overrides: dict[str | None, str]) -> pd.DataFrame:
    """Load FSD50K split and map each patch to its clip's labels."""
    frame = pd.read_csv(csv_path)
    if 'split' not in frame.columns:
        frame['split'] = None
    frame['split_name'] = frame['split'].map(split_overrides)
    frame = frame[frame['split_name'].notna()].copy()

    # Ensure fname is a string
    frame['fname_str'] = frame['fname'].astype(str)

    # Get all patch files from disk
    patch_files = sorted(data_root.glob('*.npy'))

    # Parse patch files: extract clip ID (everything before last underscore)
    patch_map = {}  # clip_id -> list of patch paths
    for patch_file in patch_files:
        # Filename format: {clip_id}_{offset:06d}.npy
        parts = patch_file.stem.rsplit('_', 1)
        if len(parts) == 2:
            clip_id = parts[0]
            if clip_id not in patch_map:
                patch_map[clip_id] = []
            patch_map[clip_id].append(str(patch_file))

    print(f'\n--- Debugging load_fsd50k_split_patches for {csv_path.name} ---')
    print(f'Found {len(patch_map)} unique clips with patches')
    print(f'Clip IDs from CSV: {frame["fname_str"].nunique()}')

    # Expand frame: each patch gets its own row with the clip's label
    expanded_rows = []
    for _, row in frame.iterrows():
        clip_id = row['fname_str']
        if clip_id in patch_map:
            for patch_path in patch_map[clip_id]:
                expanded_rows.append({
                    'fname': row['fname'],
                    'labels': row['labels'],
                    'mids': row['mids'],
                    'split_name': row['split_name'],
                    'data_path': patch_path,
                    'target': labels_to_multihot(row['labels'])
                })

    frame = pd.DataFrame(expanded_rows)

    if len(frame) == 0:
        print(f'WARNING: No patches found for {csv_path.name}')
    else:
        print(f'Expanded to {len(frame)} patch examples from {len(expanded_rows)} clips')

    print('--- End Debugging ---\n')
    return frame[['fname', 'labels', 'mids', 'split_name', 'data_path', 'target']]


vocabulary_df = load_fsd50k_vocabulary(FSD50K_GROUND_TRUTH_DIR / 'vocabulary.csv')
LABEL_ORDER = vocabulary_df['label'].tolist()
LABEL_TO_INDEX = {label: index for index, label in enumerate(LABEL_ORDER)}

print(f'Loaded {len(LABEL_ORDER)} audio classes')

Loaded 200 audio classes


In [6]:
# Load train/validation/test splitswith patches
train_validation_df = load_fsd50k_split_patches(
    FSD50K_GROUND_TRUTH_DIR / 'dev.csv',
    FSD50K_FEATURE_DIR,
    split_overrides={'train': 'train', 'val': 'validation'},
)

test_df = load_fsd50k_split_patches(
    FSD50K_GROUND_TRUTH_DIR / 'eval.csv',
    EVAL_NPY_FSD50K_DIR,
    split_overrides={None: 'test'},
)

fsd50k_df = pd.concat([train_validation_df, test_df], ignore_index=True)
assert not fsd50k_df.empty, 'No FSD50K patch examples were found.'
assert fsd50k_df['target'].map(lambda target: target.sum() > 0).all(), 'Every patch must have at least one label.'

split_summary = fsd50k_df.groupby('split_name').size().reindex(['train', 'validation', 'test'])
label_frequency = np.stack(fsd50k_df['target'].to_numpy()).sum(axis=0)

print({
    'num_classes': len(LABEL_ORDER),
    'split_counts': split_summary.fillna(0).astype(int).to_dict(),
    'available_clips': int(len(fsd50k_df)),
    'nonzero_labels': int(np.count_nonzero(label_frequency)),
})
if test_df.empty:
    print('⚠️ WARNING: No eval patches detected. Training/validation will run, but test evaluation will be skipped.')

print('\nSplit summary:')
print(split_summary)


--- Debugging load_fsd50k_split_patches for dev.csv ---
Found 40966 unique clips with patches
Clip IDs from CSV: 40966
Expanded to 527797 patch examples from 527797 clips
--- End Debugging ---


--- Debugging load_fsd50k_split_patches for eval.csv ---
Found 10231 unique clips with patches
Clip IDs from CSV: 10231
Expanded to 186456 patch examples from 186456 clips
--- End Debugging ---

{'num_classes': 200, 'split_counts': {'train': 458850, 'validation': 68947, 'test': 186456}, 'available_clips': 714253, 'nonzero_labels': 200}

Split summary:
split_name
train         458850
validation     68947
test          186456
dtype: int64


## Step 5: Audio Processing Functions

In [7]:
def load_feature_array(feature_path: tf.Tensor) -> tf.Tensor:
    path = feature_path.numpy().decode('utf-8')
    feature = np.load(path)
    feature = np.asarray(feature, dtype=np.float32)

    # Normalise the array shape so downstream resizing always sees a 2D mel image.
    if feature.ndim == 1:
        feature = np.expand_dims(feature, axis=0)
    if feature.ndim != 2:
        feature = np.squeeze(feature)
        if feature.ndim != 2:
            raise ValueError(f'Unexpected feature shape: {feature.shape} for {path}')

    expected_mels = PREPROCESSING_CONFIG['mel']['n_mels']
    if feature.shape[0] == expected_mels:
        return feature
    if feature.shape[1] == expected_mels:
        return feature.T

    raise ValueError(
        f'Unexpected mel shape {feature.shape} for {path}; expected one axis to be {expected_mels} bins.'
    )


def mel_feature_to_image(feature: tf.Tensor) -> tf.Tensor:
    feature = tf.convert_to_tensor(feature, dtype=tf.float32)
    feature = tf.expand_dims(feature, axis=-1)
    feature = tf.image.resize(feature, IMAGE_SIZE)
    feature = tf.image.per_image_standardization(feature)
    return feature


def load_example(data_path: tf.Tensor, target: tf.Tensor):
    feature = tf.py_function(load_feature_array, [data_path], Tout=tf.float32)
    feature.set_shape([None, None])
    image = mel_feature_to_image(feature)
    image.set_shape([IMAGE_SIZE[0], IMAGE_SIZE[1], 1])
    target.set_shape([len(LABEL_ORDER)])
    return image, target

print('Numpy-feature processing functions defined')

Numpy-feature processing functions defined


## Step 6: Create Datasets

In [8]:
def make_dataset(split_name: str, training: bool) -> tf.data.Dataset:
    frame = fsd50k_df[fsd50k_df['split_name'] == split_name].copy()
    assert not frame.empty, f'No examples available for split: {split_name}'
    targets = np.stack(frame['target'].to_numpy()).astype(np.float32)
    # Changed 'feature_path' to 'data_path' as per load_fsd50k_split modification
    dataset = tf.data.Dataset.from_tensor_slices((frame['data_path'].to_numpy(), targets))

    if training:
        dataset = dataset.shuffle(buffer_size=len(frame), seed=42, reshuffle_each_iteration=True)
        options = tf.data.Options()
        options.experimental_deterministic = False
        dataset = dataset.with_options(options)

    dataset = dataset.map(lambda data_path, target: load_example(data_path, target), num_parallel_calls=AUTOTUNE)
    dataset = dataset.batch(BATCH_SIZE, drop_remainder=training).prefetch(AUTOTUNE)
    return dataset

train_ds = make_dataset('train', training=True)
validation_ds = make_dataset('validation', training=False)
test_ds = make_dataset('test', training=False) if (fsd50k_df['split_name'] == 'test').any() else None

print('Datasets created')
print(f'Train dataset: {train_ds}')
print(f'Validation dataset: {validation_ds}')
print(f'Test dataset: {test_ds}')

Datasets created
Train dataset: <_PrefetchDataset element_spec=(TensorSpec(shape=(32, 96, 32, 1), dtype=tf.float32, name=None), TensorSpec(shape=(32, 200), dtype=tf.float32, name=None))>
Validation dataset: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 96, 32, 1), dtype=tf.float32, name=None), TensorSpec(shape=(None, 200), dtype=tf.float32, name=None))>
Test dataset: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 96, 32, 1), dtype=tf.float32, name=None), TensorSpec(shape=(None, 200), dtype=tf.float32, name=None))>


## Step 7: Build Model

In [9]:
class BinaryFocalCrossentropy(keras.losses.Loss):
    """Focal loss down-weights easy examples to focus on hard positive/negative classes."""
    def __init__(self, gamma: float = 2.0, alpha: float = 0.25, name: str = 'binary_focal_crossentropy'):
        super().__init__(name=name)
        self.gamma = gamma
        self.alpha = alpha

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        # Clip predictions to prevent NaN log values
        epsilon = keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)

        # Calculate focal loss components
        bce = -y_true * tf.math.log(y_pred) - (1.0 - y_true) * tf.math.log(1.0 - y_pred)
        p_t = y_true * y_pred + (1.0 - y_true) * (1.0 - y_pred)
        focal_modulation = tf.math.pow(1.0 - p_t, self.gamma)

        loss = focal_modulation * bce

        if self.alpha >= 0:
            alpha_t = y_true * self.alpha + (1.0 - y_true) * (1.0 - self.alpha)
            loss = alpha_t * loss

        return tf.reduce_mean(loss, axis=-1)

def build_mobilenetv4_classifier(num_classes: int) -> tuple[keras.Model, keras.Model]:
    inputs = keras.Input(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 1), name='log_mel')
    backbone = mobilenet.MobileNet(
        model_id=MODEL_ID,
        input_specs=keras.layers.InputSpec(shape=[None, IMAGE_SIZE[0], IMAGE_SIZE[1], 1]),
        stochastic_depth_drop_rate=0.1,
        output_stride=32,
        name='mobilenetv4_backbone',
    )
    backbone.trainable = True
    endpoints = backbone(inputs)
    final_endpoint_key = sorted([key for key in endpoints if key.isdigit()], key=int)[-1]
    x = endpoints[final_endpoint_key]
    if len(x.shape) == 4:
        x = keras.layers.GlobalAveragePooling2D(name='global_avg_pool')(x)
    x = keras.layers.Dropout(0.2, name='pre_embedding_dropout')(x)
    embedding = keras.layers.Dense(EMBEDDING_DIM, name='embedding')(x)
    normalized_embedding = keras.layers.Lambda(
        lambda tensor: tf.math.l2_normalize(tensor, axis=-1),
        name='embedding_norm',
    )(embedding)
    outputs = keras.layers.Dense(
        num_classes,
        activation='sigmoid',
        dtype='float32',
        name='fsd50k_multilabel_classifier',
    )(normalized_embedding)
    classifier = keras.Model(inputs=inputs, outputs=outputs, name='soundpulse_mobilenetv4_fsd50k_classifier')
    embedding_model = keras.Model(inputs=inputs, outputs=normalized_embedding, name='soundpulse_mobilenetv4_embedding_model')
    return classifier, embedding_model

classifier, embedding_model = build_mobilenetv4_classifier(len(LABEL_ORDER))
classifier.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY),
    loss=BinaryFocalCrossentropy(gamma=2.0, alpha=0.25),
    metrics=[
        keras.metrics.BinaryAccuracy(name='binary_accuracy', threshold=PREDICTION_THRESHOLD),
        keras.metrics.AUC(name='micro_auprc', curve='PR', multi_label=False),
    ],
)
print('Model built and compiled')
print(f'Model ID: {MODEL_ID}')

Model built and compiled
Model ID: MobileNetV4ConvSmall


In [10]:
import numpy as np
import tf_keras as keras  # Use the tf_keras package explicitly
from sklearn.metrics import precision_recall_curve, auc

class MacroAUPRCCallback(keras.callbacks.Callback):
    def __init__(self, validation_dataset):
        super().__init__()
        x_batches = []
        y_batches = []
        for x, y in validation_dataset:
            x_batches.append(x.numpy())
            y_batches.append(y.numpy())

        self.X_val = np.concatenate(x_batches, axis=0)
        self.y_val = np.concatenate(y_batches, axis=0)

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        preds = self.model.predict(self.X_val, verbose=0)

        num_classes = self.y_val.shape[1]
        auprcs = []
        for i in range(num_classes):
            precision, recall, _ = precision_recall_curve(self.y_val[:, i], preds[:, i])
            auprcs.append(auc(recall, precision))

        macro_auprc = np.mean(auprcs)
        logs['val_macro_auprc'] = macro_auprc
        print(f" - val_macro_auprc: {macro_auprc:.4f}")

# How to use it:
# 1. Instantiate callback
macro_auprc = MacroAUPRCCallback(validation_ds)

2026-06-05 19:29:43.745235: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


## Step 8: Train Model

In [11]:
callbacks = [
    macro_auprc,
    keras.callbacks.ModelCheckpoint(
        filepath=str(MODEL_OUTPUT_DIR / 'best.weights.h5'),
        monitor='val_macro_auprc',
        mode='max',
        save_best_only=True,
        save_weights_only=True,
    ),
    keras.callbacks.EarlyStopping(monitor='val_macro_auprc', mode='max', patience=4, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_macro_auprc', mode='max', factor=0.5, patience=2, min_lr=1e-6),
]

history = classifier.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

print('\n✓ Training completed!')

Epoch 1/20
14339/14339 [==============================] - 897s 62ms/step - loss: 0.0068 - binary_accuracy: 0.9878 - micro_auprc: 0.2039 - val_loss: 0.0053 - val_binary_accuracy: 0.9854 - val_micro_auprc: 0.3578 - val_macro_auprc: 0.1541 - lr: 3.0000e-04
Epoch 2/20
14339/14339 [==============================] - 988s 69ms/step - loss: 0.0032 - binary_accuracy: 0.9897 - micro_auprc: 0.5938 - val_loss: 0.0054 - val_binary_accuracy: 0.9855 - val_micro_auprc: 0.3711 - val_macro_auprc: 0.1853 - lr: 3.0000e-04
Epoch 3/20
14339/14339 [==============================] - 1175s 82ms/step - loss: 0.0028 - binary_accuracy: 0.9907 - micro_auprc: 0.6718 - val_loss: 0.0055 - val_binary_accuracy: 0.9859 - val_micro_auprc: 0.3822 - val_macro_auprc: 0.2027 - lr: 3.0000e-04
Epoch 4/20
14339/14339 [==============================] - 1057s 74ms/step - loss: 0.0026 - binary_accuracy: 0.9914 - micro_auprc: 0.7196 - val_loss: 0.0057 - val_binary_accuracy: 0.9859 - val_micro_auprc: 0.3743 - val_macro_auprc: 0.2051

## Step 9: Evaluate Model

### Calculate lwrap

In [12]:
import sklearn.metrics
# Calculate the overall lwlrap using sklearn.metrics function.

def calculate_overall_lwlrap_sklearn(truth, scores):
  """Calculate the overall lwlrap using sklearn.metrics.lrap."""
  # sklearn doesn't correctly apply weighting to samples with no labels, so just skip them.
  sample_weight = np.sum(truth > 0, axis=1)
  nonzero_weight_sample_indices = np.flatnonzero(sample_weight > 0)
  overall_lwlrap = sklearn.metrics.label_ranking_average_precision_score(
      truth[nonzero_weight_sample_indices, :] > 0,
      scores[nonzero_weight_sample_indices, :],
      sample_weight=sample_weight[nonzero_weight_sample_indices])
  return overall_lwlrap

In [13]:

if test_ds is None:
    metrics = {'status': 'skipped', 'reason': 'No eval .npy features found'}
    with (MODEL_OUTPUT_DIR / 'fsd50k_test_metrics.json').open('w', encoding='utf-8') as handle:
        json.dump(metrics, handle, indent=2)
    print('Test evaluation skipped: no test dataset available.')
else:
    test_probabilities = classifier.predict(test_ds)
    test_targets = np.concatenate([targets.numpy() for _, targets in test_ds], axis=0)
    test_predictions = (test_probabilities >= PREDICTION_THRESHOLD).astype(np.float32)

    # Compute Multilabel confusion matrix
    mcm = multilabel_confusion_matrix(test_targets, test_predictions)

    metrics = {
        'micro_f1': float(f1_score(test_targets, test_predictions, average='micro', zero_division=0)),
        'macro_f1': float(f1_score(test_targets, test_predictions, average='macro', zero_division=0)),
        'micro_average_precision': float(average_precision_score(test_targets, test_probabilities, average='micro')),
        'macro_average_precision': float(average_precision_score(test_targets, test_probabilities, average='macro')),
        'label_ranking_average_precision': float(label_ranking_average_precision_score(test_targets, test_probabilities)),
        'label_weighted_ranking_average_precision': calculate_overall_lwlrap_sklearn(test_targets, test_probabilities),
        'hamming_loss': float(hamming_loss(test_targets, test_predictions)),
        'Multilabel Confusion Matrix': np.sum(mcm, axis=0).tolist()
    }

    with (MODEL_OUTPUT_DIR / 'fsd50k_test_metrics.json').open('w', encoding='utf-8') as handle:
        json.dump(metrics, handle, indent=2)

    print('Test Metrics:')
    for metric, value in metrics.items():
      if metric == 'Multilabel Confusion Matrix':
          print(f'{metric}:\n{np.array(value)}')  # Pretty prints it back as an array block
      else:
          print(f'{metric}: {value:.4f}')

5827/5827 [==============================] - 62s 10ms/step


2026-06-05 23:29:36.531505: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Test Metrics:
micro_f1: 0.2580
macro_f1: 0.1310
micro_average_precision: 0.3287
macro_average_precision: 0.2183
label_ranking_average_precision: 0.4488
label_weighted_ranking_average_precision: 0.4458
hamming_loss: 0.0193
Multilabel Confusion Matrix:
[[36444990    52217]
 [  668661   125332]]


In [15]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create a DataFrame for easier plotting
label_counts_df = pd.DataFrame({
    'label': LABEL_ORDER,
    'frequency': label_frequency
})

# Sort by frequency to easily see underrepresented classes
label_counts_df = label_counts_df.sort_values('frequency', ascending=True)

#View classes with the lowest labels
print(label_counts_df.head(20))

ModuleNotFoundError: No module named 'seaborn'

In [16]:
#View classes with the lowest labels

print(label_counts_df.tail(20))

NameError: name 'label_counts_df' is not defined

## Step 10: Save Model Artifacts

In [17]:
# Collect embeddings
def collect_embeddings(dataset: tf.data.Dataset) -> tuple[np.ndarray, np.ndarray]:
    embeddings = []
    targets = []
    for batch_images, batch_targets in dataset:
        embeddings.append(embedding_model.predict(batch_images, verbose=0))
        targets.append(batch_targets.numpy())
    return np.concatenate(embeddings, axis=0), np.concatenate(targets, axis=0)

if test_ds is None:
    print('Embedding export skipped: no test dataset available.')
else:
    test_embeddings, test_embedding_targets = collect_embeddings(test_ds)
    np.save(MODEL_OUTPUT_DIR / 'test_embeddings.npy', test_embeddings)
    np.save(MODEL_OUTPUT_DIR / 'test_targets.npy', test_embedding_targets)

    print(f'Test embeddings shape: {test_embeddings.shape}')

Test embeddings shape: (186456, 512)


In [18]:
# Save models
classifier.save(MODEL_OUTPUT_DIR / 'classifier.keras')
embedding_model.save(MODEL_OUTPUT_DIR / 'embedding_model.keras')
embedding_model.export(str(MODEL_OUTPUT_DIR / 'saved_model'))

# Convert to TFLite
converter = tf.lite.TFLiteConverter.from_saved_model(str(MODEL_OUTPUT_DIR / 'saved_model'))
# Add the following line to allow TensorFlow operations (FlexDelegate)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
tflite_model = converter.convert()
with (MODEL_OUTPUT_DIR / 'mobilenetv4_conv_small_embedding.tflite').open('wb') as handle:
    handle.write(tflite_model)

# Save metadata
metadata = {
    'model_name': MODEL_ID,
    'training_mode': 'from_scratch',
    'dataset': 'FSD50K',
    'task': 'multi_label_audio_classification',
    'num_classes': len(LABEL_ORDER),
    'label_order': LABEL_ORDER,
    'embedding_dim': EMBEDDING_DIM,
    'image_size': IMAGE_SIZE,
    'sample_rate_hz': SAMPLE_RATE,
    'window_duration_ms': WINDOW_DURATION_MS,
    'prediction_threshold': PREDICTION_THRESHOLD,
}
with (MODEL_OUTPUT_DIR / 'export_metadata.json').open('w', encoding='utf-8') as handle:
    json.dump(metadata, handle, indent=2)

print(f'✓ Models saved to {MODEL_OUTPUT_DIR}')

INFO:tensorflow:Assets written to: /Users/oselumese/Documents/GitHub/SoundPulse/ml/export/artifacts/generated/mobilenetv4_convsmall/saved_model/assets


INFO:tensorflow:Assets written to: /Users/oselumese/Documents/GitHub/SoundPulse/ml/export/artifacts/generated/mobilenetv4_convsmall/saved_model/assets


Saved artifact at '/Users/oselumese/Documents/GitHub/SoundPulse/ml/export/artifacts/generated/mobilenetv4_convsmall/saved_model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 96, 32, 1), dtype=tf.float32, name='log_mel')
Output Type:
  TensorSpec(shape=(None, 512), dtype=tf.float16, name=None)
Captures:
  5482153552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5482154512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5482154704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5482152016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5482157200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5482156624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5482167952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5482152976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5482155664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5482155280: TensorSpec(shape=(), dtype=tf.

W0000 00:00:1780699028.833701  269558 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1780699028.833899  269558 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
2026-06-05 23:37:08.840806: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /Users/oselumese/Documents/GitHub/SoundPulse/ml/export/artifacts/generated/mobilenetv4_convsmall/saved_model
2026-06-05 23:37:08.843949: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-06-05 23:37:08.843957: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /Users/oselumese/Documents/GitHub/SoundPulse/ml/export/artifacts/generated/mobilenetv4_convsmall/saved_model
I0000 00:00:1780699028.870848  269558 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
2026-06-05 23:37:08.874411: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-06-05 23:37:09.031661: I tensorflow/c